# Imports & Setup

In [1]:
import os, sys, gc, time, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

COMP_DATA = Path('/kaggle/input/competitions/stanford-rna-3d-folding-2')
PDB_RNA_DIR = COMP_DATA / 'PDB_RNA'
MSA_DIR = COMP_DATA / 'MSA'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

test_df = pd.read_csv('/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv')
test_df['seq_len'] = test_df['sequence'].apply(len)
print(f"Test sequences: {len(test_df)}")
print(test_df[['target_id','seq_len']].head(10))

Device: cpu
Test sequences: 28
  target_id  seq_len
0      8ZNQ       30
1      9IWF       69
2      9JGM      210
3      9MME     4640
4      9J09      214
5      9E9Q      101
6      9CFN       59
7      9OBM       73
8      9G4P       68
9      9G4Q      104


# Helper Functions

In [2]:
def parse_fasta(text):
    seqs, cur_hdr, cur_seq = {}, None, []
    for line in text.strip().split('\n'):
        line = line.strip()
        if line.startswith('>'):
            if cur_hdr: seqs[cur_hdr] = ''.join(cur_seq)
            cur_hdr, cur_seq = line[1:], []
        elif line:
            cur_seq.append(line)
    if cur_hdr: seqs[cur_hdr] = ''.join(cur_seq)
    return seqs

def generate_ideal_helix(L, rise=2.81, radius=9.5, twist_deg=32.7):
    coords = np.zeros((L, 3))
    twist = np.radians(twist_deg)
    for i in range(L):
        coords[i,0] = radius * np.cos(i * twist)
        coords[i,1] = radius * np.sin(i * twist)
        coords[i,2] = i * rise
    return coords.astype(np.float32)

def interpolate_coords(coords):
    L = len(coords)
    valid = ~np.isnan(coords[:,0])
    if not valid.any(): return generate_ideal_helix(L)
    if valid.all(): return coords
    result = coords.copy()
    valid_idx = np.where(valid)[0]
    for i in range(L):
        if valid[i]: continue
        left = valid_idx[valid_idx < i]
        right = valid_idx[valid_idx > i]
        if len(left) > 0 and len(right) > 0:
            l, r = left[-1], right[0]
            t = (i - l) / (r - l)
            result[i] = (1-t)*result[l] + t*result[r]
        elif len(left) > 0:
            step = result[left[-1]] - result[left[-2]] if len(left)>1 else np.array([0,0,2.81])
            result[i] = result[left[-1]] + step*(i-left[-1])
        elif len(right) > 0:
            step = result[right[1]] - result[right[0]] if len(right)>1 else np.array([0,0,-2.81])
            result[i] = result[right[0]] + step*(i-right[0])
    return result

print("Helper functions ready")

Helper functions ready


# Load PDB Database & Build K-mer Index

In [3]:
print("Loading PDB RNA sequences...")

fasta_path = PDB_RNA_DIR / 'pdb_seqres_NA.fasta'
print(f"Reading from: {fasta_path}")

with open(fasta_path, 'r') as f:
    pdb_fasta = f.read()

pdb_seqs_raw = parse_fasta(pdb_fasta)
print(f"Loaded {len(pdb_seqs_raw):,} sequences")

# Build structured list
pdb_entries = []
for header, seq in pdb_seqs_raw.items():
    seq_clean = seq.replace('-','').upper().replace('T','U')
    if len(seq_clean) < 5:
        continue
    parts = header.split('_')
    if len(parts) >= 2:
        pdb_id = parts[0].upper()
        chain = parts[1].split()[0]
        pdb_entries.append((pdb_id, chain, seq_clean, len(seq_clean)))

print(f"Indexed {len(pdb_entries):,} PDB chains")

# Build 5-mer index
print("Building k-mer index...")
K = 5
kmer_index = defaultdict(list)
for idx, (pdb_id, chain, seq, L) in enumerate(pdb_entries):
    seen = set()
    for i in range(len(seq)-K+1):
        km = seq[i:i+K]
        if 'N' not in km and km not in seen:
            kmer_index[km].append(idx)
            seen.add(km)

print(f"K-mer index built: {len(kmer_index):,} unique {K}-mers")

Loading PDB RNA sequences...
Reading from: /kaggle/input/competitions/stanford-rna-3d-folding-2/PDB_RNA/pdb_seqres_NA.fasta
Loaded 26,255 sequences
Indexed 25,144 PDB chains
Building k-mer index...
K-mer index built: 1,363 unique 5-mers


# CIF Parser

In [4]:
_cif_cache = {}

def load_cif_coords(pdb_id):
    pdb_id = pdb_id.upper()
    if pdb_id in _cif_cache:
        return _cif_cache[pdb_id]
    
    cif_path = PDB_RNA_DIR / f'{pdb_id}.cif'
    if not cif_path.exists():
        _cif_cache[pdb_id] = {}
        return {}
    
    coords_by_chain = defaultdict(list)
    in_atom_loop = False
    col_map = {}
    col_idx = 0
    
    RESNAME_MAP = {
        'ADE':'A','GUA':'G','CYT':'C','URA':'U','THY':'T',
        'DA':'A','DG':'G','DC':'C','DT':'T','DU':'U',
        'A':'A','G':'G','C':'C','U':'U','T':'T'
    }
    
    try:
        with open(cif_path, 'r', errors='replace') as f:
            for line in f:
                line_s = line.strip()
                if line_s == 'loop_':
                    in_atom_loop = False; col_map = {}; col_idx = 0
                    continue
                if line_s.startswith('_atom_site.'):
                    field = line_s.split('.')[1].split()[0]
                    col_map[field] = col_idx; col_idx += 1
                    in_atom_loop = True
                    continue
                if in_atom_loop and line_s and not line_s.startswith('_') and not line_s.startswith('#'):
                    parts = line_s.split()
                    if not col_map or len(parts) < max(col_map.values())+1:
                        continue
                    try:
                        atom = parts[col_map['label_atom_id']]
                        if atom != "C1'": continue
                        chain = parts[col_map.get('label_asym_id', 6)]
                        resname = parts[col_map.get('label_comp_id', 5)]
                        seq_id_str = parts[col_map.get('label_seq_id', 8)]
                        if seq_id_str in ('.','?'): continue
                        x = float(parts[col_map['Cartn_x']])
                        y = float(parts[col_map['Cartn_y']])
                        z = float(parts[col_map['Cartn_z']])
                        rn = RESNAME_MAP.get(resname.strip()[:3], resname[0] if resname else 'N')
                        coords_by_chain[chain].append((int(seq_id_str), rn, x, y, z))
                    except (KeyError, ValueError, IndexError):
                        pass
                elif line_s.startswith('#') and in_atom_loop:
                    if 'label_atom_id' in col_map:
                        break
                    in_atom_loop = False; col_map = {}; col_idx = 0
    except Exception:
        pass
    
    result = {}
    for chain, recs in coords_by_chain.items():
        recs.sort(key=lambda r: r[0])
        result[chain] = recs
    
    _cif_cache[pdb_id] = result
    return result

print("CIF parser ready")

CIF parser ready


# Alignment & Template Search

In [5]:
def global_align(seq1, seq2, MAX=400):
    s1 = seq1[:MAX].upper().replace('T','U')
    s2 = seq2[:MAX].upper().replace('T','U')
    n, m = len(s1), len(s2)
    MATCH, MISMATCH, GAP = 2, -1, -2
    dp = np.zeros((n+1, m+1), dtype=np.float32)
    dp[:,0] = np.arange(n+1)*GAP
    dp[0,:] = np.arange(m+1)*GAP
    for i in range(1, n+1):
        for j in range(1, m+1):
            sc = MATCH if s1[i-1]==s2[j-1] else MISMATCH
            dp[i,j] = max(dp[i-1,j-1]+sc, dp[i-1,j]+GAP, dp[i,j-1]+GAP)
    a1, a2 = [], []
    i, j = n, m
    while i>0 or j>0:
        if i>0 and j>0:
            sc = MATCH if s1[i-1]==s2[j-1] else MISMATCH
            if abs(dp[i,j]-(dp[i-1,j-1]+sc))<0.01: a1.append(s1[i-1]); a2.append(s2[j-1]); i-=1; j-=1
            elif i>0 and abs(dp[i,j]-(dp[i-1,j]+GAP))<0.01: a1.append(s1[i-1]); a2.append('-'); i-=1
            else: a1.append('-'); a2.append(s2[j-1]); j-=1
        elif i>0: a1.append(s1[i-1]); a2.append('-'); i-=1
        else: a1.append('-'); a2.append(s2[j-1]); j-=1
    ra1=''.join(reversed(a1)); ra2=''.join(reversed(a2))
    matches = sum(c1==c2 and c1!='-' for c1,c2 in zip(ra1,ra2))
    total = sum(c1!='-' or c2!='-' for c1,c2 in zip(ra1,ra2))
    return ra1, ra2, matches/max(total,1)


def find_templates(query_seq, target_id, n_templates=5, min_identity=0.25):
    query = query_seq.upper().replace('T','U')
    query_kmers = set(query[i:i+K] for i in range(len(query)-K+1) if 'N' not in query[i:i+K])
    
    votes = defaultdict(int)
    for km in query_kmers:
        for entry_idx in kmer_index.get(km, []):
            votes[entry_idx] += 1
    
    if not votes:
        return []
    
    # MSA bonus
    msa_pdb_ids = set()
    msa_path = MSA_DIR / f'{target_id}.MSA.fasta'
    if msa_path.exists():
        try:
            with open(msa_path,'r') as f: msa_text = f.read()
            for hdr in parse_fasta(msa_text).keys():
                for part in hdr.split('|'):
                    p = part.strip()
                    if 3 <= len(p) <= 6 and p[:4].isalnum():
                        msa_pdb_ids.add(p[:4].upper())
        except: pass
    
    top_candidates = sorted(votes.items(), key=lambda x: -x[1])[:60]
    templates = []
    seen_pdbs = set()
    
    for entry_idx, vote_count in top_candidates:
        if len(templates) >= n_templates * 2: break
        pdb_id, chain, tmpl_seq, tmpl_len = pdb_entries[entry_idx]
        if pdb_id in seen_pdbs: continue
        len_ratio = tmpl_len / max(len(query),1)
        if len_ratio < 0.3 or len_ratio > 3.0: continue
        jaccard = vote_count / max(len(query_kmers),1)
        if jaccard > 0.05 or pdb_id in msa_pdb_ids:
            _, _, identity = global_align(query, tmpl_seq)
            if identity >= min_identity:
                bonus = 0.1 if pdb_id in msa_pdb_ids else 0.0
                templates.append({'pdb_id': pdb_id, 'chain': chain,
                                   'identity': identity+bonus, 'template_seq': tmpl_seq})
                seen_pdbs.add(pdb_id)
    
    templates.sort(key=lambda x: -x['identity'])
    return templates[:n_templates]


def template_to_coords(query_seq, template, perturb=0.0):
    all_chains = load_cif_coords(template['pdb_id'])
    if not all_chains: return None
    chain = template['chain']
    if chain not in all_chains:
        chain = min(all_chains.keys(), key=lambda c: abs(len(all_chains[c])-len(query_seq)))
    recs = all_chains[chain]
    if not recs: return None
    tmpl_seq = ''.join(r[1] for r in recs).replace('T','U')
    tmpl_xyz = np.array([[r[2],r[3],r[4]] for r in recs], dtype=np.float32)
    query_u = query_seq.upper().replace('T','U')
    a_q, a_t, _ = global_align(query_u, tmpl_seq)
    L = len(query_u)
    coords = np.full((L,3), np.nan, dtype=np.float32)
    q_pos = t_pos = 0
    for aq, at in zip(a_q, a_t):
        if aq!='-' and at!='-':
            if q_pos<L and t_pos<len(tmpl_xyz): coords[q_pos] = tmpl_xyz[t_pos]
        if aq!='-': q_pos+=1
        if at!='-': t_pos+=1
    coords = interpolate_coords(coords)
    if perturb > 0:
        np.random.seed(SEED + int(perturb*100))
        coords += np.random.normal(0, perturb, coords.shape).astype(np.float32)
    return coords

print("Alignment & template search ready")

Alignment & template search ready


# Predict 5 Structures Per Target

In [6]:
def predict_5(target_id, sequence):
    L = len(sequence)
    templates = find_templates(sequence, target_id, n_templates=5)
    best_id = templates[0]['identity'] if templates else 0.0
    preds = []

    # Use templates
    for i, tmpl in enumerate(templates):
        coords = template_to_coords(sequence, tmpl, perturb=0.4*i)
        if coords is not None:
            preds.append(coords)

    # Fill remaining slots
    while len(preds) < 5:
        if preds:
            np.random.seed(SEED + len(preds)*13)
            noise_scale = 1.5 * len(preds)
            preds.append(preds[0] + np.random.normal(0, noise_scale, preds[0].shape).astype(np.float32))
        else:
            twist_options = [32.7, 30.0, 35.0, 28.5, 37.5]
            preds.append(generate_ideal_helix(L, twist_deg=twist_options[len(preds)%5]))

    return preds[:5], best_id

# Quick test
row0 = test_df.iloc[0]
t0 = time.time()
preds, best_id = predict_5(row0['target_id'], row0['sequence'])
print(f"Test run: {row0['target_id']} (L={row0['seq_len']}) best_identity={best_id:.2f} in {time.time()-t0:.2f}s")
print(f"Prediction shapes: {[p.shape for p in preds]}")

Test run: 8ZNQ (L=30) best_identity=0.00 in 0.03s
Prediction shapes: [(30, 3), (30, 3), (30, 3), (30, 3), (30, 3)]


# Main Loop & Save Submission

In [7]:
print(f"Generating predictions for {len(test_df)} sequences...\n")
all_rows = []
start_time = time.time()
identity_log = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    target_id = row['target_id']
    sequence = row['sequence'].upper().replace('T','U')

    try:
        preds, best_id = predict_5(target_id, sequence)
        identity_log.append(best_id)

        for res_idx, residue in enumerate(sequence):
            row_data = {'ID': f'{target_id}_{res_idx+1}', 'resname': residue, 'resid': res_idx+1}
            for pi, pred in enumerate(preds):
                x, y, z = float(pred[res_idx,0]), float(pred[res_idx,1]), float(pred[res_idx,2])
                if np.isnan(x) or np.isnan(y) or np.isnan(z):
                    x, y, z = 0.0, 0.0, float(res_idx*2.81)
                row_data[f'x_{pi+1}'] = round(max(-999.999, min(9999.999, x)), 3)
                row_data[f'y_{pi+1}'] = round(max(-999.999, min(9999.999, y)), 3)
                row_data[f'z_{pi+1}'] = round(max(-999.999, min(9999.999, z)), 3)
            all_rows.append(row_data)

        if idx % 5 == 0:
            elapsed = time.time()-start_time
            eta = elapsed/(idx+1)*(len(test_df)-idx-1)
            print(f"[{idx+1}/{len(test_df)}] {target_id} L={len(sequence)} identity={best_id:.2f} ETA={eta/60:.1f}min")

    except Exception as e:
        print(f"ERROR {target_id}: {e}")
        for res_idx, residue in enumerate(sequence):
            row_data = {'ID': f'{target_id}_{res_idx+1}', 'resname': residue, 'resid': res_idx+1}
            for pi in range(1,6):
                row_data[f'x_{pi}'] = 0.0; row_data[f'y_{pi}'] = 0.0
                row_data[f'z_{pi}'] = round(res_idx*2.81, 3)
            all_rows.append(row_data)

print(f"\nDone in {(time.time()-start_time)/60:.1f} minutes")
print(f"Avg template identity: {np.mean(identity_log):.3f}")
print(f"Sequences with identity>0.5: {sum(i>0.5 for i in identity_log)}/{len(identity_log)}")

# Build submission
coord_cols = [f'{c}_{i}' for i in range(1,6) for c in ['x','y','z']]
submission = pd.DataFrame(all_rows)[['ID','resname','resid'] + coord_cols]
for col in coord_cols:
    submission[col] = submission[col].fillna(0.0).clip(-999.999, 9999.999)

print(f"\nSubmission shape: {submission.shape}")
print(submission.head())
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f"\n✓ Saved submission.csv ({os.path.getsize('/kaggle/working/submission.csv')/1e6:.1f} MB)")

Generating predictions for 28 sequences...



  0%|          | 0/28 [00:00<?, ?it/s]

[1/28] 8ZNQ L=30 identity=0.00 ETA=0.0min
[6/28] 9E9Q L=101 identity=1.00 ETA=0.1min
[11/28] 9G4R L=47 identity=1.00 ETA=0.1min
[16/28] 9I9W L=28 identity=0.00 ETA=0.1min
[21/28] 9WHV L=80 identity=0.00 ETA=0.0min
[26/28] 9EBP L=81 identity=1.00 ETA=0.0min

Done in 0.1 minutes
Avg template identity: 0.644
Sequences with identity>0.5: 18/28

Submission shape: (9762, 18)
       ID resname  resid    x_1    y_1    z_1    x_2     y_2     z_2     x_3  \
0  8ZNQ_1       A      1  9.500  0.000   0.00  7.064  -0.153  -2.715   6.667   
1  8ZNQ_2       C      2  7.994  5.132   2.81  8.388   5.522   2.238  10.508   
2  8ZNQ_3       C      3  3.955  8.638   5.62  3.951   9.150   6.966   5.168   
3  8ZNQ_4       G      4 -1.339  9.405   8.43 -1.880  11.890   6.646   0.039   
4  8ZNQ_5       U      5 -6.207  7.191  11.24 -3.708   4.186  10.523  -4.826   

      y_3     z_3    x_4     y_4     z_4     x_5     y_5     z_5  
0   0.600  -4.621  3.380  -5.559   0.270  13.961   9.127  -4.241  
1   6.317   3